# Map — Proposed EV Charging Stations (Spain 2027)

Self-contained interactive HTML map of every segment the pipeline proposes as a new-charger location. Driven by `charger_counts.csv` (output of `Regression.ipynb`), which in turn was driven by `optimal_locations_top.csv` (output of `Classification.ipynb`).

### Competition-compliance notes
- **No installation** required to view.
- **No login** / no external services.
- **No live internet**: uses Plotly's built-in natural-earth vector outline (no tile server). `include_plotlyjs='inline'` embeds the Plotly.js library inside the HTML file.
- Colour-coded markers, hover popups with route / probability / charger count / demand kW / region.

### Known placeholder
Until the downstream Outputs notebook computes the real `grid_status` (`Sufficient | Moderate | Congested`) from i-DE / Endesa / Viesgo substation capacity, markers are colour-coded by **quartile of `prob_needs_station`** as a stand-in: green (lowest probability → most confidently okay) → red (highest → urgent). This is flagged in the map's title and legend.

---
## 1 · Setup

In [36]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

---
## 2 · Load predictions

In [37]:
DATA_DIR = "C:/Users/vldma/Datathon/Datasets/Modelling datasets"
df = pd.read_csv(f"{DATA_DIR}/charger_counts.csv")
print("loaded:", df.shape)
df.head()

loaded: (81, 8)


,segment_id,carretera,comunidad_autonoma,road_centroid_lat,road_centroid_lon,prob_needs_station,n_chargers_proposed,estimated_demand_kw
0,1025,N-401,08 - C.La Mancha,39.001170,-3.928287,0.991121,12,1800
1,705,N-211,02 - Aragón,40.890687,-0.769281,0.990548,9,1350
2,733,N-232A,02 - Aragón,41.166500,-0.267939,0.989595,1,150
3,734,N-232A,02 - Aragón,41.166500,-0.267939,0.989595,1,150
4,702,N-204,unknown,40.603296,-3.247541,0.989011,7,1050


### 2.1 Placeholder colour band (quartile of `prob_needs_station`)

Replaces the real `grid_status` until substation capacity is joined in. Lowest probability → green; highest → red.

In [38]:
# qcut on prob_needs_station with 4 bins, lowest → least urgent
df["colour_band"] = pd.qcut(
    df["prob_needs_station"],
    q=4,
    labels=["Low (placeholder)", "Medium-low", "Medium-high", "High (placeholder)"],
)
print(df["colour_band"].value_counts().sort_index())

colour_band
Low (placeholder)     21
Medium-low            20
Medium-high           20
High (placeholder)    20
Name: count, dtype: int64


---
## 3 · Plotly scatter_geo

In [39]:
fig = px.scatter_geo(
    df,
    lat="road_centroid_lat",
    lon="road_centroid_lon",
    size="n_chargers_proposed",
    size_max=18,
    color="colour_band",
    color_discrete_map={
        "Low (placeholder)":  "#2ca02c",
        "Medium-low":         "#ffdd00",
        "Medium-high":        "#ff7f0e",
        "High (placeholder)": "#d62728",
    },
    hover_name="carretera",
    hover_data={
        "segment_id": True,
        "prob_needs_station": ":.3f",
        "n_chargers_proposed": True,
        "estimated_demand_kw": True,
        "comunidad_autonoma": True,
        "colour_band": False,
        "road_centroid_lat": False,
        "road_centroid_lon": False,
    },
    projection="natural earth",
    scope="europe",
    title="Proposed EV Charging Stations — Spain 2027 "
          "(colour = placeholder urgency quartile until grid_status is computed)",
)

fig.update_geos(
    lataxis_range=[35.5, 44],
    lonaxis_range=[-10, 5],
    showcountries=True,
    showcoastlines=True,
    coastlinecolor="#888",
    countrycolor="#888",
    landcolor="#f7f7f2",
    oceancolor="#cfe7f5",
    showocean=True,
)
fig.update_layout(
    height=800,
    legend=dict(title="Urgency (placeholder)", orientation="v", x=0.02, y=0.98),
    margin=dict(l=0, r=0, t=60, b=0),
)
fig.show()

---
## 4 · Export self-contained HTML

`include_plotlyjs='inline'` embeds ~3 MB of Plotly.js in the file — no internet, no install, just double-click to open.

In [40]:
OUT_PATH = "C:/Users/vldma/Datathon/ev_station_map.html"
fig.write_html(OUT_PATH, include_plotlyjs="inline", full_html=True)

import os
size_mb = os.path.getsize(OUT_PATH) / 1e6
print(f"wrote {OUT_PATH}  ({size_mb:.2f} MB)")
print("double-click the file to open offline in any browser.")

wrote C:/Users/vldma/Datathon/ev_station_map.html  (4.58 MB)
double-click the file to open offline in any browser.


---
## 5 · Combined map — existing & per-charger proposals

Shows every existing interurban station **and every proposed charger** as a distinct marker.

**Proposed-location geometry (approximation).** The dataset gives one centroid per *road*, not per segment. We estimate segment positions by:
1. Sorting segments within each road by `segment_id` (the raw dataset assigns these sequentially along the route).
2. Computing a cumulative-length fraction for each segment along its parent road.
3. Interpolating between the road's bounding-box corners using that fraction → per-segment lat/lon.
4. Within each segment, distributing its `n_chargers_proposed` markers evenly along the segment's portion of the bounding-box diagonal.

Each resulting marker is **one proposed charger** (= the minimum count the regression predicted to meet 2027 demand for that location). Markers are colour-coded by urgency quartile (placeholder for `grid_status`).

In [41]:
# 1. Existing stations
existing = pd.read_csv(f"{DATA_DIR}/m2_charging_sites_interurban.csv")
existing = existing.dropna(subset=["latitude", "longitude"])
print("existing stations:", existing.shape)

# 2. Raw segment metadata (length + road bounding box) for proposed-location estimation
raw = pd.read_csv(f"{DATA_DIR}/ev_charging_segment_ml_dataset.csv")
raw.columns = [c.lower().strip().replace(" ", "_") for c in raw.columns]
meta_cols = [
    "segment_id", "length_m", "road_total_length_m",
    "road_min_lat", "road_max_lat", "road_min_lon", "road_max_lon",
]

df_full = df.merge(raw[meta_cols], on="segment_id", how="left")
print("proposed segments (charger_counts × raw metadata):", df_full.shape)

existing stations: (3679, 11)
proposed segments (charger_counts × raw metadata): (81, 15)


### 5.1 Estimate per-segment coordinates

In [42]:
# Estimate each segment's centroid by interpolating along the parent road's bounding box,
# using cumulative segment length as the position fraction.
df_full = df_full.sort_values(["carretera", "segment_id"]).reset_index(drop=True)

cum_len = df_full.groupby("carretera")["length_m"].cumsum()
df_full["cum_len_mid"] = cum_len - df_full["length_m"] / 2
df_full["frac"] = (df_full["cum_len_mid"] / df_full["road_total_length_m"].clip(lower=1)).clip(0, 1)

df_full["seg_lat"] = df_full["road_min_lat"] + df_full["frac"] * (df_full["road_max_lat"] - df_full["road_min_lat"])
df_full["seg_lon"] = df_full["road_min_lon"] + df_full["frac"] * (df_full["road_max_lon"] - df_full["road_min_lon"])

# Fallbacks if bbox was missing
df_full["seg_lat"] = df_full["seg_lat"].fillna(df_full["road_centroid_lat"])
df_full["seg_lon"] = df_full["seg_lon"].fillna(df_full["road_centroid_lon"])

print("sample segment coords (first 5 on first road):")
print(df_full.head(5)[["carretera", "segment_id", "length_m", "frac", "seg_lat", "seg_lon"]])

sample segment coords (first 5 on first road):
  carretera  segment_id      length_m      frac    seg_lat   seg_lon
0      A-12          75  73728.056493  0.326190  42.737934 -1.747212
1      A-23         128  59162.621123  0.074615  39.857730 -1.204644
2      A-23         133  63129.982289  0.228849  40.232776 -1.057406
3      A-23         137     15.635710  0.308488  40.426431 -0.981380
4      A-40         196  80025.767788  0.222291  40.001109 -3.806960


### 5.2 Explode into individual chargers and spread along segment

In [43]:
# Explode each segment into n_chargers_proposed individual charger rows,
# then spread the chargers along the segment's portion of the road bbox diagonal.
chargers = df_full.loc[df_full.index.repeat(df_full["n_chargers_proposed"])].reset_index(drop=True)
chargers["charger_idx"] = chargers.groupby("segment_id").cumcount()
n_per_seg = chargers.groupby("segment_id")["charger_idx"].transform("count").astype(int)

# Local road direction (bbox diagonal) + segment span in degrees
dlat = chargers["road_max_lat"] - chargers["road_min_lat"]
dlon = chargers["road_max_lon"] - chargers["road_min_lon"]
road_deg = np.sqrt(dlat ** 2 + dlon ** 2).replace(0, np.nan)
unit_lat = (dlat / road_deg).fillna(0)
unit_lon = (dlon / road_deg).fillna(0)
seg_deg = (chargers["length_m"] / chargers["road_total_length_m"].clip(lower=1)) * road_deg.fillna(0)
seg_deg = seg_deg.clip(upper=0.25)  # cap ≈ 25 km spread per segment so markers stay on-road

# Charger offset fraction within segment: -0.5 … +0.5, 0 when n=1
offset_frac = (chargers["charger_idx"] + 0.5) / n_per_seg - 0.5
chargers["lat"] = chargers["seg_lat"] + offset_frac * seg_deg * unit_lat
chargers["lon"] = chargers["seg_lon"] + offset_frac * seg_deg * unit_lon

print(f"total individual chargers proposed: {len(chargers)}")
print("sample exploded rows:")
chargers[["carretera", "segment_id", "charger_idx", "n_chargers_proposed", "lat", "lon", "colour_band"]].head(10)

total individual chargers proposed: 422
sample exploded rows:


,carretera,segment_id,charger_idx,n_chargers_proposed,lat,lon,colour_band
0,A-12,75,0,8,42.714729,-1.773894,Medium-high
1,A-12,75,1,8,42.721359,-1.766271,Medium-high
2,A-12,75,2,8,42.727989,-1.758647,Medium-high
3,A-12,75,3,8,42.734619,-1.751024,Medium-high
4,A-12,75,4,8,42.741249,-1.743400,Medium-high
5,A-12,75,5,8,42.747879,-1.735776,Medium-high
6,A-12,75,6,8,42.754509,-1.728153,Medium-high
7,A-12,75,7,8,42.761139,-1.720529,Medium-high
8,A-23,128,0,7,39.757997,-1.243798,Medium-low
9,A-23,128,1,7,39.791241,-1.230747,Medium-low


### 5.3 Combined map (existing dots + one marker per proposed charger)

In [44]:
fig2 = go.Figure()

# Layer 1 — existing stations (small steel-blue dots)
fig2.add_trace(go.Scattergeo(
    lat=existing["latitude"],
    lon=existing["longitude"],
    mode="markers",
    name=f"Existing ({len(existing):,})",
    marker=dict(size=4, color="#1f77b4", opacity=0.55, line=dict(width=0)),
    text=existing.apply(
        lambda r: (f"<b>{r.get('name','')}</b><br>"
                   f"operator: {r.get('operator','')}<br>"
                   f"max power: {r.get('max_power_kw','')} kW<br>"
                   f"road: {r.get('Carretera','')}"), axis=1),
    hovertemplate="%{text}<extra></extra>",
))

# Layers 2-5 — proposed chargers (one marker per charger), grouped by urgency band
colour_map = {
    "Low (placeholder)":  "#2ca02c",
    "Medium-low":         "#ffdd00",
    "Medium-high":        "#ff7f0e",
    "High (placeholder)": "#d62728",
}
for band, colour in colour_map.items():
    sub = chargers[chargers["colour_band"] == band]
    if sub.empty:
        continue
    fig2.add_trace(go.Scattergeo(
        lat=sub["lat"],
        lon=sub["lon"],
        mode="markers",
        name=f"Proposed — {band} ({len(sub)})",
        marker=dict(size=7, color=colour, opacity=0.9, line=dict(width=1, color="black"),
                    symbol="circle"),
        text=sub.apply(
            lambda r: (f"<b>{r['carretera']}</b>  (segment {r['segment_id']})<br>"
                       f"charger {int(r['charger_idx'])+1}/{int(r['n_chargers_proposed'])} at location<br>"
                       f"prob needs station: {r['prob_needs_station']:.3f}<br>"
                       f"150 kW fast charger<br>"
                       f"segment total demand: {r['estimated_demand_kw']} kW<br>"
                       f"region: {r['comunidad_autonoma']}"), axis=1),
        hovertemplate="%{text}<extra></extra>",
    ))

fig2.update_geos(
    projection_type="natural earth",
    scope="europe",
    lataxis_range=[35.5, 44],
    lonaxis_range=[-10, 5],
    showcountries=True,
    showcoastlines=True,
    coastlinecolor="#888",
    countrycolor="#888",
    landcolor="#f7f7f2",
    oceancolor="#cfe7f5",
    showocean=True,
)
fig2.update_layout(
    title=(f"Existing ({len(existing):,}) vs. Proposed individual chargers ({len(chargers):,}) — Spain 2027"),
    height=820,
    legend=dict(title="Layer", orientation="v", x=0.02, y=0.98,
                bgcolor="rgba(255,255,255,0.8)"),
    margin=dict(l=0, r=0, t=60, b=0),
)
fig2.show()

---
## 6 · Export combined map

In [45]:
OUT_PATH2 = "C:/Users/vldma/Datathon/ev_station_map_combined.html"
fig2.write_html(OUT_PATH2, include_plotlyjs="inline", full_html=True)

import os
size_mb = os.path.getsize(OUT_PATH2) / 1e6
print(f"wrote {OUT_PATH2}  ({size_mb:.2f} MB)")
print("open offline in any browser by double-click.")

wrote C:/Users/vldma/Datathon/ev_station_map_combined.html  (5.36 MB)
open offline in any browser by double-click.
